# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah200401/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Method choice:

I will use Logistic Regression as the first model for the Refresh / Content Opportunity Scoring lane.

Logistic Regression is appropriate because the task is to rank content by the likelihood of a defined refresh-opportunity outcome. It is simple, interpretable, and provides a useful baseline for understanding which signals contribute to the ranking.

I prefer this model over a more complex model at this stage because the goal is to establish an honest, explainable model that can be compared with the Week-4 rule. More complexity is not automatically better.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [15]:
# Section 1: Method choice

from sklearn.linear_model import LogisticRegression

# We choose Logistic Regression because it is:
# 1. Simple
# 2. Interpretable
# 3. Suitable for a binary classification task
# 4. A good first model to compare against the Week-4 rule

model_type = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("=== METHOD CHOICE ===")
print("Model: Logistic Regression")
print("Reason: Simple, interpretable, and suitable as a first ML model.")
print("Goal: Compare the ML model against the Week-4 baseline.")

=== METHOD CHOICE ===
Model: Logistic Regression
Reason: Simple, interpretable, and suitable as a first ML model.
Goal: Compare the ML model against the Week-4 baseline.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### 2. Split design

I will use a grouped train/test split by client so that content from the same client does not appear in both training and test sets.

This is more honest for this dataset because multiple content items can belong to the same client and may share client-specific patterns.

The client identifier will be used only for grouping the split, not as a model feature. I will keep the test set completely separate from model fitting and use the same evaluation metric as the Week-4 baseline.


In [2]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_Token")
login(token=HF_TOKEN)

content_df = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)["train"].to_pandas()

print("✅ content_df loaded")
print("Rows:", len(content_df))

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

✅ content_df loaded
Rows: 519606


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### 3) Train + compare vs my baseline

I will train the selected model using the training portion only and evaluate it on the held-out test portion.

The model will be compared against the Week-4 baseline using the same evaluation metric and evaluation data.

I will focus on whether the model provides a meaningful improvement over the simple rule-based baseline rather than assuming that a more complex model is automatically better.


In [4]:
from sklearn.model_selection import GroupShuffleSplit

# Create modeling dataframe
model_df = content_df.copy()

# Remove rows without client ID
model_df = model_df.dropna(subset=["client_hash_id"]).copy()

# Grouped split by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("✅ Split created")
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print("Client overlap:", len(overlap))

if len(overlap) == 0:
    print("✅ No client leakage")

✅ Split created
Train rows: 439038
Test rows: 80568
Train clients: 67
Test clients: 17
Client overlap: 0
✅ No client leakage


In [6]:
# Check that the required dataframes exist

print("content_df:", "content_df" in globals())
print("train_df:", "train_df" in globals())
print("test_df:", "test_df" in globals())

if "train_df" in globals() and "test_df" in globals():
    print("\n✅ Train/test split exists.")
else:
    print("\n❌ Train/test split is missing. Run Section 2 split cell first.")

content_df: True
train_df: True
test_df: True

✅ Train/test split exists.


In [9]:
# Check possible outcome / performance columns

possible_columns = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "impressions_last30",
    "clicks_last30",
    "impressions_prev30",
    "clicks_prev30",
    "avg_position_90d",
    "avg_position_last30",
    "avg_position_prev30"
]

available = [c for c in possible_columns if c in train_df.columns]

print("Available columns:")
for c in available:
    print("-", c)

Available columns:
- search_volume


In [10]:
from datasets import load_dataset

query_df = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    token=HF_TOKEN
)["train"].to_pandas()

print("✅ Query table loaded")
print("Rows:", len(query_df))
print("\nColumns:")
print(query_df.columns.tolist())

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

✅ Query table loaded
Rows: 2414248

Columns:
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


In [11]:
# Section 3: Check outcome fields

outcome_cols = [
    "impressions_90d",
    "clicks_90d",
    "impressions_last30",
    "clicks_last30",
    "impressions_prev30",
    "clicks_prev30",
    "avg_position_90d",
    "avg_position_last30",
    "avg_position_prev30"
]

available_outcomes = [
    c for c in outcome_cols
    if c in query_df.columns
]

print("Available outcome columns:")
for c in available_outcomes:
    print("-", c)

Available outcome columns:
- impressions_90d
- clicks_90d
- impressions_last30
- clicks_last30
- impressions_prev30
- clicks_prev30
- avg_position_90d
- avg_position_last30
- avg_position_prev30


In [12]:
# Section 3: Build modeling dataset

import pandas as pd
import numpy as np

# Select model features from the query-performance table.
# These describe observed search demand/performance.
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d",
    "content_visible_query_count",
    "rare_query_count",
    "rare_impressions_share",
    "anonymized_impressions_share"
]

feature_cols = [
    c for c in feature_cols
    if c in query_df.columns
]

# Target: recent clicks
target_col = "clicks_last30"

# Keep required columns
model_df = query_df[
    ["client_hash_id", "content_hash_id"] +
    feature_cols +
    [target_col]
].copy()

# Remove rows where target is missing
model_df = model_df.dropna(subset=[target_col])

# Create binary target:
# 1 = content received at least one click in the last 30 days
# 0 = no clicks in the last 30 days
model_df["target"] = (
    model_df[target_col] > 0
).astype(int)

print("✅ Modeling dataset created")
print("Rows:", len(model_df))
print("Features:", feature_cols)

print("\nTarget distribution:")
print(model_df["target"].value_counts())

print("\nTarget percentage:")
print(model_df["target"].value_counts(normalize=True).round(4))

✅ Modeling dataset created
Rows: 2414248
Features: ['impressions_90d', 'clicks_90d', 'avg_position_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']

Target distribution:
target
0    2329133
1      85115
Name: count, dtype: int64

Target percentage:
target
0    0.9647
1    0.0353
Name: proportion, dtype: float64


In [13]:
# ============================================================
# SECTION 3: TRAIN + EVALUATE LOGISTIC REGRESSION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# 1. Prepare modeling dataset
# ------------------------------------------------------------

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d",
    "content_visible_query_count",
    "rare_query_count",
    "rare_impressions_share",
    "anonymized_impressions_share"
]

feature_cols = [c for c in feature_cols if c in query_df.columns]

model_df = query_df[
    ["client_hash_id", "content_hash_id"] +
    feature_cols +
    ["clicks_last30"]
].copy()

model_df = model_df.dropna(subset=["clicks_last30"])

# Binary target
model_df["target"] = (
    model_df["clicks_last30"] > 0
).astype(int)

print("✅ Modeling dataset created")
print("Rows:", len(model_df))
print("Features:", feature_cols)

# ------------------------------------------------------------
# 2. Grouped train/test split by client
# ------------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df["target"],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("\n=== SPLIT ===")
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

overlap = (
    set(train_df["client_hash_id"])
    & set(test_df["client_hash_id"])
)

print("Client overlap:", len(overlap))

# ------------------------------------------------------------
# 3. Train Logistic Regression
# ------------------------------------------------------------

X_train = train_df[feature_cols]
y_train = train_df["target"]

X_test = test_df[feature_cols]
y_test = test_df["target"]

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

# ------------------------------------------------------------
# 4. Predictions
# ------------------------------------------------------------

pred = model.predict(X_test)

print("\n=== LOGISTIC REGRESSION ===")
print(
    "Precision:",
    round(
        precision_score(
            y_test,
            pred,
            zero_division=0
        ),
        4
    )
)

print(
    "Recall:",
    round(
        recall_score(
            y_test,
            pred,
            zero_division=0
        ),
        4
    )
)

print(
    "F1:",
    round(
        f1_score(
            y_test,
            pred,
            zero_division=0
        ),
        4
    )
)

print("\n=== CLASSIFICATION REPORT ===")
print(
    classification_report(
        y_test,
        pred,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 5. Feature interpretation
# ------------------------------------------------------------

classifier = model.named_steps["classifier"]

coefficients = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": classifier.coef_[0]
})

coefficients["abs_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "abs_coefficient",
    ascending=False
)

print("\n=== FEATURE IMPORTANCE ===")
print(
    coefficients[
        ["feature", "coefficient"]
    ].to_string(index=False)
)

print("\n✅ Section 3 model training complete")

✅ Modeling dataset created
Rows: 2414248
Features: ['impressions_90d', 'clicks_90d', 'avg_position_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']

=== SPLIT ===
Train rows: 2164186
Test rows: 250062
Train clients: 41
Test clients: 11
Client overlap: 0

=== LOGISTIC REGRESSION ===
Precision: 0.7361
Recall: 0.3136
F1: 0.4398

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.98      1.00      0.99    241174
           1       0.74      0.31      0.44      8888

    accuracy                           0.97    250062
   macro avg       0.86      0.65      0.71    250062
weighted avg       0.97      0.97      0.97    250062


=== FEATURE IMPORTANCE ===
                     feature  coefficient
                  clicks_90d     3.535472
            avg_position_90d    -0.757726
            rare_query_count     0.350858
 content_visible_query_count    -0.323678
      rare_

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### 4) Errors and interpretation

I will inspect the model's incorrect predictions and the most influential features.

The goal is to understand where the model makes mistakes and whether its learned signals are reasonable for the content refresh task.

I will treat these errors as useful evidence about the limitations of the current feature set rather than assuming that a more complex model would automatically solve them.


In [14]:
# ============================================================
# SECTION 4: ERRORS AND INTERPRETATION
# ============================================================

# 1. Find incorrect predictions
error_df = test_df.copy()

error_df["actual"] = y_test.values
error_df["predicted"] = pred

error_df["correct"] = (
    error_df["actual"] == error_df["predicted"]
)

errors = error_df[
    error_df["correct"] == False
].copy()

print("=== MODEL ERRORS ===")
print("Total test rows:", len(error_df))
print("Incorrect predictions:", len(errors))
print(
    "Error rate:",
    round(len(errors) / len(error_df), 4)
)

# Show examples of errors
print("\n=== SAMPLE ERRORS ===")

error_columns = [
    "content_hash_id",
    "client_hash_id",
    "actual",
    "predicted"
]

error_columns += [
    c for c in feature_cols
    if c in errors.columns
]

print(
    errors[error_columns]
    .head(10)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 2. Feature interpretation
# ------------------------------------------------------------

print("\n=== FEATURE INTERPRETATION ===")

print(
    coefficients[
        ["feature", "coefficient"]
    ].to_string(index=False)
)

print("\nInterpretation:")

for _, row in coefficients.iterrows():
    direction = (
        "increases"
        if row["coefficient"] > 0
        else "decreases"
    )

    print(
        f"- {row['feature']}: "
        f"{direction} the predicted probability "
        f"of the positive class."
    )

# ------------------------------------------------------------
# 3. Basic error summary
# ------------------------------------------------------------

false_positive = (
    (error_df["actual"] == 0) &
    (error_df["predicted"] == 1)
).sum()

false_negative = (
    (error_df["actual"] == 1) &
    (error_df["predicted"] == 0)
).sum()

print("\n=== ERROR TYPES ===")
print("False positives:", false_positive)
print("False negatives:", false_negative)


=== MODEL ERRORS ===
Total test rows: 250062
Incorrect predictions: 7100
Error rate: 0.0284

=== SAMPLE ERRORS ===
         content_hash_id          client_hash_id  actual  predicted  impressions_90d  clicks_90d  avg_position_90d  content_visible_query_count  rare_query_count  rare_impressions_share  anonymized_impressions_share
content_3e512b141dbab791 client_0fa64a184f18a4a0       1          0               26           1          3.461538                          381              5491                0.074203                      0.378199
content_3e512b141dbab791 client_0fa64a184f18a4a0       1          0              447           1          3.548098                          381              5491                0.074203                      0.378199
content_3e512b141dbab791 client_0fa64a184f18a4a0       1          0               14           1          1.500000                          381              5491                0.074203                      0.378199
content_3e512b141dbab

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.